# Shortcut Learning in Risky-Intent Classification

This notebook is the main experiment runner for the project.

The goal is to study whether RoBERTa relies too much on shortcut-trigger words such as **die**, **kill**, **cut**, **jump**, and **hurt**, instead of understanding full context such as negation, temporal meaning, figurative language, or ambiguity.

This notebook runs:

- E0: TF-IDF baseline
- E1: RoBERTa baseline
- E2: RoBERTa + Keyword Masking
- E3: RoBERTa + Counterfactual Augmentation
- E4: RoBERTa + Counterfactual Augmentation + Keyword Masking
- E5: RoBERTa-CF-KM-CR final model
- E6: MC Dropout uncertainty analysis

This section is for running the project on a fresh GPU instance.

If this notebook is already inside the repository, skip the clone command and only run installation.

## 1. Clone Repository and Setup

In [ ]:
# If running on a fresh GPU instance, run this cell.
# If you are already inside the repository, skip the git clone line.

!git clone https://github.com/juliairsalina/shortcut-learning-risky-intent.git
%cd shortcut-learning-risky-intent
!pip install -r requirements.txt

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Training may be slow.")

## 2. Project Imports and File Check

This section checks whether the required data files and source scripts exist before running the experiments.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 50)

In [ ]:
required_files = [
    "data/raw/dataset_nad.xlsx",
    "data/raw/custom_ood_set_150_julia.csv",
    "src/config.py",
    "src/data_utils.py",
    "src/train_roberta.py",
    "src/evaluate.py",
]

for file_path in required_files:
    path = Path(file_path)
    if path.exists():
        print(f"FOUND   : {file_path}")
    else:
        print(f"MISSING : {file_path}")

In [ ]:
def load_json_safe(path):
    path = Path(path)
    if not path.exists():
        print(f"WARNING: Missing JSON file: {path}")
        return None
    
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_csv_safe(path):
    path = Path(path)
    if not path.exists():
        print(f"WARNING: Missing CSV file: {path}")
        return None
    
    return pd.read_csv(path)


def display_json_safe(path):
    data = load_json_safe(path)
    if data is None:
        return None
    
    print(path)
    display(pd.json_normalize(data))
    return data


def display_csv_safe(path, n=10):
    df = load_csv_safe(path)
    if df is None:
        return None
    
    print(path)
    print("Shape:", df.shape)
    display(df.head(n))
    return df


def show_label_distribution(df, name):
    if df is None:
        return
    
    print(f"\n{name} shape:", df.shape)
    
    if "label" in df.columns:
        print(f"{name} label distribution:")
        display(df["label"].value_counts(dropna=False).reset_index().rename(
            columns={"index": "label", "label": "count"}
        ))
    elif "gold_label" in df.columns:
        print(f"{name} gold_label distribution:")
        display(df["gold_label"].value_counts(dropna=False).reset_index().rename(
            columns={"index": "gold_label", "gold_label": "count"}
        ))
    else:
        print(f"{name}: No label or gold_label column found.")
    
    if "category" in df.columns:
        print(f"{name} category distribution:")
        display(df["category"].value_counts(dropna=False).reset_index().rename(
            columns={"index": "category", "category": "count"}
        ))


def show_wrong_examples(path, n=10):
    df = load_csv_safe(path)
    if df is None:
        return None
    
    label_col = None
    pred_col = None
    
    for col in ["label", "gold_label", "true_label", "y_true"]:
        if col in df.columns:
            label_col = col
            break
    
    for col in ["prediction", "pred_label", "y_pred"]:
        if col in df.columns:
            pred_col = col
            break
    
    if label_col is None or pred_col is None:
        print(f"Cannot identify label/prediction columns in {path}")
        display(df.head(n))
        return df
    
    wrong_df = df[df[label_col] != df[pred_col]].copy()
    
    print(f"Wrong examples from {path}: {len(wrong_df)}")
    
    if "confidence" in wrong_df.columns:
        wrong_df = wrong_df.sort_values("confidence", ascending=False)
        print("Sorted by confidence.")
    
    display(wrong_df.head(n))
    return wrong_df


def show_confident_wrong_examples(path, n=10):
    df = load_csv_safe(path)
    if df is None:
        return None
    
    label_col = None
    pred_col = None
    
    for col in ["label", "gold_label", "true_label", "y_true"]:
        if col in df.columns:
            label_col = col
            break
    
    for col in ["prediction", "pred_label", "y_pred"]:
        if col in df.columns:
            pred_col = col
            break
    
    if label_col is None or pred_col is None:
        print(f"Cannot identify label/prediction columns in {path}")
        display(df.head(n))
        return df
    
    wrong_df = df[df[label_col] != df[pred_col]].copy()
    
    if "confidence" in wrong_df.columns:
        wrong_df = wrong_df.sort_values("confidence", ascending=False)
        print(f"Top confident wrong examples from {path}:")
        display(wrong_df.head(n))
    else:
        print(f"No confidence column found in {path}. Showing wrong examples instead.")
        display(wrong_df.head(n))
    
    return wrong_df


def display_pngs(folder):
    folder = Path(folder)
    
    if not folder.exists():
        print(f"WARNING: Missing image folder: {folder}")
        return
    
    png_files = sorted(folder.glob("*.png"))
    
    if not png_files:
        print(f"No PNG files found in {folder}")
        return
    
    for img_path in png_files:
        print(img_path)
        display(Image(filename=str(img_path)))

## 3. Data Preparation

This section creates the processed train, validation, test, and OOD files.

The main dataset is:

`data/raw/dataset_nad.xlsx`

Expected columns:

- id
- text
- keyword
- label

The OOD dataset is:

`data/raw/custom_ood_set_150_julia.csv`

Expected columns:

- text
- keyword
- label or gold_label
- category

In [ ]:
!python -m src.data_utils

In [ ]:
train_df = display_csv_safe("data/processed/train.csv")
val_df = display_csv_safe("data/processed/val.csv")
test_df = display_csv_safe("data/processed/test.csv")
ood_df = display_csv_safe("data/processed/ood.csv")

In [ ]:
show_label_distribution(train_df, "Train")
show_label_distribution(val_df, "Validation")
show_label_distribution(test_df, "Test")
show_label_distribution(ood_df, "OOD")

## 4. Create Mitigation Datasets

This section creates additional training datasets for shortcut-learning mitigation.

Keyword masking replaces shortcut-trigger words with a mask token so the model cannot rely only on the trigger word.

Counterfactual augmentation creates contrastive examples where the shortcut keyword may stay similar, but the label changes based on context.

Expected generated files:

- `data/processed/train_masked.csv`
- `data/processed/train_counterfactual.csv`
- `data/processed/train_full.csv`

In [ ]:
!python -m src.masking
!python -m src.augmentation

In [ ]:
mitigation_files = {
    "Original train": "data/processed/train.csv",
    "Keyword masked train": "data/processed/train_masked.csv",
    "Counterfactual train": "data/processed/train_counterfactual.csv",
    "Full mitigation train": "data/processed/train_full.csv",
}

row_counts = []

for name, path in mitigation_files.items():
    df = load_csv_safe(path)
    if df is not None:
        row_counts.append({
            "dataset": name,
            "path": path,
            "rows": len(df),
            "columns": len(df.columns),
        })
    else:
        row_counts.append({
            "dataset": name,
            "path": path,
            "rows": None,
            "columns": None,
        })

display(pd.DataFrame(row_counts))

In [ ]:
for name, path in mitigation_files.items():
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    display_csv_safe(path, n=5)

## 5. Experiment E0: TF-IDF Baseline

E0 is the simple machine learning baseline using **TF-IDF + Logistic Regression**.

This baseline helps us compare RoBERTa against a simpler keyword-sensitive model.

In [ ]:
!python -m src.train_tfidf --experiment_id E0

In [ ]:
e0_metrics = display_json_safe("results/metrics/E0_tfidf.json")

In [ ]:
e0_id_preds = display_csv_safe("results/predictions/E0_id_predictions.csv", n=10)
e0_ood_preds = display_csv_safe("results/predictions/E0_ood_predictions.csv", n=10)

In [ ]:
show_wrong_examples("results/predictions/E0_ood_predictions.csv", n=10)

## 6. Experiment E1: RoBERTa Baseline

E1 is the main RoBERTa baseline trained on the original training data.

This experiment tests how well RoBERTa performs without any shortcut-learning mitigation.

Model output should be saved under:

`saved_models/E1/`

In [ ]:
!python -m src.train_roberta --experiment_id E1 --train_file data/processed/train.csv

In [ ]:
!python -m src.evaluate --experiment_id E1

In [ ]:
e1_metrics = display_json_safe("results/metrics/E1.json")

In [ ]:
e1_id_preds = display_csv_safe("results/predictions/E1_id_predictions.csv", n=10)
e1_ood_preds = display_csv_safe("results/predictions/E1_ood_predictions.csv", n=10)

In [ ]:
show_confident_wrong_examples("results/predictions/E1_ood_predictions.csv", n=10)

## 7. Experiment E2: RoBERTa + Keyword Masking

E2 tests whether keyword masking reduces shortcut-trigger word reliance.

The model is trained on:

`data/processed/train_masked.csv`

Model output should be saved under:

`saved_models/E2/`

In [ ]:
!python -m src.train_roberta --experiment_id E2 --train_file data/processed/train_masked.csv

In [ ]:
!python -m src.evaluate --experiment_id E2

In [ ]:
e2_metrics = display_json_safe("results/metrics/E2.json")

In [ ]:
e2_id_preds = display_csv_safe("results/predictions/E2_id_predictions.csv", n=10)
e2_ood_preds = display_csv_safe("results/predictions/E2_ood_predictions.csv", n=10)

In [ ]:
show_wrong_examples("results/predictions/E2_ood_predictions.csv", n=10)

## 8. Experiment E3: RoBERTa + Counterfactual Augmentation

E3 tests whether counterfactual augmentation improves context understanding.

The model is trained on:

`data/processed/train_counterfactual.csv`

Model output should be saved under:

`saved_models/E3/`

In [ ]:
!python -m src.train_roberta --experiment_id E3 --train_file data/processed/train_counterfactual.csv

In [ ]:
!python -m src.evaluate --experiment_id E3

In [ ]:
e3_metrics = display_json_safe("results/metrics/E3.json")

In [ ]:
e3_id_preds = display_csv_safe("results/predictions/E3_id_predictions.csv", n=10)
e3_ood_preds = display_csv_safe("results/predictions/E3_ood_predictions.csv", n=10)

In [ ]:
show_wrong_examples("results/predictions/E3_ood_predictions.csv", n=10)

## 8a. Experiment E4: RoBERTa + Counterfactual Augmentation + Keyword Masking

E4 tests whether combining counterfactual augmentation and keyword masking improves OOD robustness before adding confidence regularization.

The model is trained on:

`data/processed/train_full.csv`

Unlike E5, this experiment does not use confidence regularization.

Model output should be saved under:

`saved_models/E4/`

In [ ]:
!python -m src.train_roberta --experiment_id E4 --train_file data/processed/train_full.csv

In [ ]:
!python -m src.evaluate --experiment_id E4

In [ ]:
e4_metrics = display_json_safe("results/metrics/E4.json")

In [ ]:
e4_id_preds = display_csv_safe("results/predictions/E4_id_predictions.csv", n=10)
e4_ood_preds = display_csv_safe("results/predictions/E4_ood_predictions.csv", n=10)

In [ ]:
show_confident_wrong_examples("results/predictions/E4_ood_predictions.csv", n=10)

## 9. Experiment E5: Final Model RoBERTa-CF-KM-CR

E5 is the final proposed model:

**RoBERTa + Counterfactual Augmentation + Keyword Masking + Confidence Regularization**

This model is trained on:

`data/processed/train_full.csv`

Confidence regularization is enabled with:

`lambda_conf = 0.05`

Model output should be saved under:

`saved_models/E5/`

In [ ]:
!python -m src.train_roberta --experiment_id E5 --train_file data/processed/train_full.csv --confidence_regularization true --lambda_conf 0.05

In [ ]:
!python -m src.evaluate --experiment_id E5

In [ ]:
e5_metrics = display_json_safe("results/metrics/E5.json")

In [ ]:
e5_id_preds = display_csv_safe("results/predictions/E5_id_predictions.csv", n=10)
e5_ood_preds = display_csv_safe("results/predictions/E5_ood_predictions.csv", n=10)

In [ ]:
show_confident_wrong_examples("results/predictions/E5_ood_predictions.csv", n=10)

## 10. Compare Evaluation Results

This section compares E0, E1, E2, E3, and E5.

The most important metrics are:

- OOD macro-F1
- ID-OOD performance gap
- Confident wrong predictions
- Shortcut sensitivity

The final model should not be selected based only on ID accuracy.

In [ ]:
def flatten_dict(d, parent_key="", sep="_"):
    items = []
    
    if not isinstance(d, dict):
        return {}
    
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    
    return dict(items)


def get_first_available(flat_metrics, possible_keys):
    for key in possible_keys:
        if key in flat_metrics:
            return flat_metrics[key]
    return None


def count_confident_wrong(pred_path):
    df = load_csv_safe(pred_path)
    if df is None:
        return None
    
    label_col = None
    pred_col = None
    
    for col in ["label", "gold_label", "true_label", "y_true"]:
        if col in df.columns:
            label_col = col
            break
    
    for col in ["prediction", "pred_label", "y_pred"]:
        if col in df.columns:
            pred_col = col
            break
    
    if label_col is None or pred_col is None:
        return None
    
    wrong_df = df[df[label_col] != df[pred_col]].copy()
    
    if "confidence" in wrong_df.columns:
        return len(wrong_df[wrong_df["confidence"] >= 0.80])
    
    return len(wrong_df)


def load_experiment_summary(experiment_id, metric_path, pred_path):
    metrics = load_json_safe(metric_path)
    flat = flatten_dict(metrics) if metrics is not None else {}
    
    id_accuracy = get_first_available(flat, [
        "id_accuracy", "test_accuracy", "accuracy_id", "id_acc", "test_acc"
    ])
    
    id_macro_f1 = get_first_available(flat, [
        "id_macro_f1", "test_macro_f1", "macro_f1_id", "id_f1", "test_f1"
    ])
    
    ood_accuracy = get_first_available(flat, [
        "ood_accuracy", "accuracy_ood", "ood_acc"
    ])
    
    ood_macro_f1 = get_first_available(flat, [
        "ood_macro_f1", "macro_f1_ood", "ood_f1"
    ])
    
    id_ood_gap = None
    if id_macro_f1 is not None and ood_macro_f1 is not None:
        try:
            id_ood_gap = float(id_macro_f1) - float(ood_macro_f1)
        except:
            id_ood_gap = None
    
    confident_wrong_count = get_first_available(flat, [
        "confident_wrong_count", "ood_confident_wrong_count"
    ])
    
    if confident_wrong_count is None:
        confident_wrong_count = count_confident_wrong(pred_path)
    
    return {
        "experiment_id": experiment_id,
        "id_accuracy": id_accuracy,
        "id_macro_f1": id_macro_f1,
        "ood_accuracy": ood_accuracy,
        "ood_macro_f1": ood_macro_f1,
        "id_ood_gap": id_ood_gap,
        "confident_wrong_count": confident_wrong_count,
    }

In [ ]:
comparison_rows = [
    load_experiment_summary(
        "E0",
        "results/metrics/E0_tfidf.json",
        "results/predictions/E0_ood_predictions.csv",
    ),
    load_experiment_summary(
        "E1",
        "results/metrics/E1.json",
        "results/predictions/E1_ood_predictions.csv",
    ),
    load_experiment_summary(
        "E2",
        "results/metrics/E2.json",
        "results/predictions/E2_ood_predictions.csv",
    ),
    load_experiment_summary(
        "E3",
        "results/metrics/E3.json",
        "results/predictions/E3_ood_predictions.csv",
    ),
    load_experiment_summary(
        "E4",
        "results/metrics/E4.json",
        "results/predictions/E4_ood_predictions.csv",
    ),
    load_experiment_summary(
        "E5",
        "results/metrics/E5.json",
        "results/predictions/E5_ood_predictions.csv",
    ),
]
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

In [ ]:
print("Best model checks:")

if "ood_macro_f1" in comparison_df.columns and comparison_df["ood_macro_f1"].notna().any():
    best_ood = comparison_df.sort_values("ood_macro_f1", ascending=False).head(1)
    print("\nHighest OOD macro-F1:")
    display(best_ood)

if "id_ood_gap" in comparison_df.columns and comparison_df["id_ood_gap"].notna().any():
    best_gap = comparison_df.sort_values("id_ood_gap", ascending=True).head(1)
    print("\nLowest ID-OOD gap:")
    display(best_gap)

if "confident_wrong_count" in comparison_df.columns and comparison_df["confident_wrong_count"].notna().any():
    best_cw = comparison_df.sort_values("confident_wrong_count", ascending=True).head(1)
    print("\nLowest confident wrong count:")
    display(best_cw)

## 11. Shortcut Sensitivity Analysis

This analysis checks whether model predictions change when shortcut keywords are masked.

If a model depends too much on shortcut words, its predicted probability may change strongly when the keyword is replaced with `[MASK]`.

Important outputs:

- `shortcut_summary.csv`
- `keyword_sensitivity_details.csv`

In [ ]:
!python -m src.shortcut_metrics

In [ ]:
shortcut_summary = display_csv_safe("results/shortcut_metrics/shortcut_summary.csv", n=20)
keyword_details = display_csv_safe("results/shortcut_metrics/keyword_sensitivity_details.csv", n=10)

In [ ]:
if keyword_details is not None:
    if "abs_probability_change" in keyword_details.columns:
        print("Top 10 examples with highest absolute probability change:")
        display(keyword_details.sort_values("abs_probability_change", ascending=False).head(10))
    elif "abs_prob_change" in keyword_details.columns:
        print("Top 10 examples with highest absolute probability change:")
        display(keyword_details.sort_values("abs_prob_change", ascending=False).head(10))
    else:
        print("No absolute probability change column found.")
        display(keyword_details.head(10))

In [ ]:
if keyword_details is not None:
    flip_cols = ["prediction_flipped", "flipped", "is_flipped"]
    flip_col = None
    
    for col in flip_cols:
        if col in keyword_details.columns:
            flip_col = col
            break
    
    if flip_col is not None:
        print("Examples where prediction flipped after keyword masking:")
        display(keyword_details[keyword_details[flip_col] == True].head(10))
    else:
        print("No prediction flip column found.")

## 12. MC Dropout Uncertainty Analysis

MC Dropout is used as an inference-time uncertainty baseline, not as the final model.

The idea is to run the model multiple times with dropout enabled during inference.

This can estimate uncertainty using entropy, variance, or disagreement across predictions.

Here, we compare:

- E1 RoBERTa baseline
- E5 final RoBERTa-CF-KM-CR model

In [ ]:
!python -m src.mc_dropout --experiment_id E1 --mc_passes 30
!python -m src.mc_dropout --experiment_id E5 --mc_passes 30

In [ ]:
e1_uncertainty_summary = display_json_safe("results/uncertainty/E1_mc_dropout_summary.json")
e5_uncertainty_summary = display_json_safe("results/uncertainty/E5_mc_dropout_summary.json")

In [ ]:
e1_uncertainty_ood = display_csv_safe("results/uncertainty/E1_mc_dropout_ood.csv", n=10)
e5_uncertainty_ood = display_csv_safe("results/uncertainty/E5_mc_dropout_ood.csv", n=10)

In [ ]:
uncertainty_rows = []

for exp_id, path in [
    ("E1", "results/uncertainty/E1_mc_dropout_summary.json"),
    ("E5", "results/uncertainty/E5_mc_dropout_summary.json"),
]:
    data = load_json_safe(path)
    if data is not None:
        row = {"experiment_id": exp_id}
        row.update(flatten_dict(data))
        uncertainty_rows.append(row)

uncertainty_comparison_df = pd.DataFrame(uncertainty_rows)
display(uncertainty_comparison_df)

In [ ]:
for exp_id, df in [
    ("E1", e1_uncertainty_ood),
    ("E5", e5_uncertainty_ood),
]:
    if df is None:
        continue
    
    print("\n" + "=" * 80)
    print(f"{exp_id} MC Dropout examples")
    print("=" * 80)
    
    if "entropy" in df.columns:
        print("Highest entropy examples:")
        display(df.sort_values("entropy", ascending=False).head(10))
    
    if "variance" in df.columns:
        print("Highest variance examples:")
        display(df.sort_values("variance", ascending=False).head(10))
    elif "prob_variance" in df.columns:
        print("Highest probability variance examples:")
        display(df.sort_values("prob_variance", ascending=False).head(10))
    
    label_col = None
    pred_col = None
    
    for col in ["label", "gold_label", "true_label", "y_true"]:
        if col in df.columns:
            label_col = col
            break
    
    for col in ["prediction", "pred_label", "y_pred"]:
        if col in df.columns:
            pred_col = col
            break
    
    if label_col is not None and pred_col is not None:
        wrong_df = df[df[label_col] != df[pred_col]].copy()
        
        if "confidence" in wrong_df.columns:
            print("Confident wrong examples:")
            display(wrong_df.sort_values("confidence", ascending=False).head(10))
        else:
            print("Wrong examples:")
            display(wrong_df.head(10))

## 13. SHAP Analysis

SHAP is used as a small interpretability case study to check whether the baseline focuses more on shortcut keywords than the final model.

We compare:

- E1 RoBERTa baseline
- E5 final RoBERTa-CF-KM-CR model

SHAP may be slow and can be skipped if GPU time is limited.

In [ ]:
!python -m src.shap_analysis --experiment_id E1
!python -m src.shap_analysis --experiment_id E5

In [ ]:
shap_summary = display_csv_safe("results/shap/shap_summary.csv", n=20)
e1_shap_reliance = display_csv_safe("results/shap/E1_shap_keyword_reliance.csv", n=10)
e5_shap_reliance = display_csv_safe("results/shap/E5_shap_keyword_reliance.csv", n=10)

In [ ]:
display_pngs("results/shap")

## 14. Generate Final Plots

This section generates report-ready plots.

Expected plots may include:

- `id_vs_ood_f1.png`
- `id_ood_gap.png`
- `keyword_sensitivity.png`
- `confident_wrong_count.png`
- `shap_keyword_reliance.png`

If a plot does not exist, the notebook will not crash.

In [ ]:
!python -m src.plot_utils

In [ ]:
display_pngs("results/plots")

# Final Result Summary

This section collects the most important report-ready outputs.

Main notes:

- E1 is the baseline RoBERTa model.
- E2 tests keyword masking.
- E3 tests counterfactual augmentation.
- E5 is the final proposed model.
- MC Dropout is an uncertainty baseline, not the final model.
- SHAP is used for qualitative and quantitative keyword reliance analysis.
- The best model should be selected based on OOD macro-F1, ID-OOD gap, keyword sensitivity, and confident wrong count, not only ID accuracy.

In [ ]:
print("Main model comparison table")
display(comparison_df)

In [ ]:
print("Shortcut sensitivity summary")

shortcut_summary = load_csv_safe("results/shortcut_metrics/shortcut_summary.csv")

if shortcut_summary is not None:
    display(shortcut_summary)
else:
    print("Shortcut sensitivity summary not available.")

In [ ]:
print("MC Dropout uncertainty summary")

if not uncertainty_comparison_df.empty:
    display(uncertainty_comparison_df)
else:
    print("MC Dropout uncertainty summary not available.")

In [ ]:
print("SHAP summary")

shap_summary = load_csv_safe("results/shap/shap_summary.csv")

if shap_summary is not None:
    display(shap_summary)
else:
    print("SHAP summary not available.")

In [ ]:
print("SHAP summary")

shap_summary = load_csv_safe("results/shap/shap_summary.csv")

if shap_summary is not None:
    display(shap_summary)
else:
    print("SHAP summary not available.")

In [ ]:
print("Final plots")
display_pngs("results/plots")

In [ ]:
print("SHAP plots")
display_pngs("results/shap")